# Notebook 02 — Human vs LLM Extraction Validation (Tiers 1–3)

**Purpose:** Three-tier validation of the LLM extraction against human references:
- **Tier 1:** Corpus-wide comparison against pre-existing manual extraction (64 trials, Group A/B fields)
- **Tier 2:** Full-schema comparison against 5-trial gold standard (re-extracted manually)
- **Tier 3:** Qualitative error pattern review of 10-trial spot check

**Data Flow:**

| Direction | File | From / To |
| :--------- | :---- | :--------- |
| Input | `data/raw/prior_extraction_clean.csv` | Pre-existing manual extraction |
| Input | `data/raw/tier2_gold_standard.jsonl` | 5-trial manual re-extraction |
| Input | `data/raw/tier3_filled.csv` | 10-trial spot-check results |
| Input | `data/raw/field_validation_mapping.csv` | Field tier + bucket labels |
| Output | `outputs/tables/tier1_per_field.csv` | Tier 1 per-field agreement |
| Output | `outputs/tables/tier1_adjudication_queue.csv` | Tier 1 disagreements for review |
| Output | `outputs/tables/tier2_per_field.csv` | Tier 2 per-field agreement |
| Output | `outputs/tables/tier2_adjudication_queue.csv` | Tier 2 disagreements for review |
| Output | `outputs/tables/tier3_error_summary.csv` | Tier 3 error pattern summary |
| Output | `outputs/tables/appendix2_combined.csv` | All-tier combined summary |

**Prerequisite notebooks:** None (standalone validation; uses the same LLM run A as N01).

**Index:**

| Section | What it does |
| :------- | :------------ |
| 0 | Setup — paths, imports |
| Tier 1 | Prior extraction comparison (15 eligible fields, 64 trials) |
| Tier 2 | 5-trial gold standard comparison (59 non-excluded fields, adjudication queue) |
| Tier 3 | 10-trial spot check error review |
| Combined | Appendix 2: All-tier summary table |

> **Shared code:** Uses `src/loaders.py` (LLM runs, prior extraction, gold
> standard), `src/normalization.py` (normalize + parse prior text), and
> `src/agreement.py` (bucket-specific agreement metrics).

**Notebook-specific notes:**

- **Adjudication queues:** Tier 1 produces `tier1_adjudication_queue.csv`
  (disagreements on Group A fields only — 14 A_reference fields where the prior
  extraction has high confidence). Tier 2 produces `tier2_adjudication_queue.csv`
  (disagreements on all non-excluded fields across the 5-trial gold standard —
  small enough to review comprehensively). Both are compiled for human review.
- **Tier 1 field selection:** 15 of 69 mapping fields are eligible: 14
  `A_reference` (high-confidence prior fields) + 1 `B_triangulation`
  (healthcare_setting). Remaining 54 are `EXCLUDED` or marked `no`.


## Section 0: Setup

In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path().resolve().parent if Path().resolve().name == "notebooks"          else Path().resolve()
sys.path.insert(0, str(ROOT))

# ── External libraries ───────────────────────────────────────────────────────
import pandas as pd

# ── Project modules ──────────────────────────────────────────────────────────
# Data loading — load LLM runs, prior extraction, gold standard, and field mapping
from src.analysis.loaders import load_extraction_run, load_mapping_table
from src.analysis.loaders import load_prior_extraction
from src.analysis.loaders import load_gold_standard
from src.analysis.data_loading import parse_structured_array

# Normalization — normalise raw values per bucket; parse prior free-text
from src.analysis.normalization import normalize_value, parse_prior_text

# Agreement — per-field agreement with bucket-specific statistics
from src.analysis.agreement import compute_agreement

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR = ROOT / "data" / "raw"
RUN_A_PATH = DATA_DIR / "copd_v11.jsonl"
MAPPING_TABLE_PATH = DATA_DIR / "field_validation_mapping.csv"
PRIOR_PATH = DATA_DIR / "prior_extraction_clean.csv"
GOLD_STANDARD_PATH = DATA_DIR / "tier2_gold_standard.jsonl"
TIER3_PATH = DATA_DIR / "tier3_filled.csv"

OUTPUT_DIR = ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "pipeline").mkdir(parents=True, exist_ok=True)

llm_run = load_extraction_run(RUN_A_PATH)
mapping = load_mapping_table(MAPPING_TABLE_PATH)
print(f"LLM Run A loaded: {len(llm_run)} arm-rows across {llm_run['cov_nr'].nunique()} trials")
n_excluded = len(mapping[mapping['data_type_bucket'] == 'EXCLUDED'])
print(f"Mapping table: {len(mapping)} field definitions ({n_excluded} needs_discussion_* pipeline helper fields excluded from all analyses)")


LLM Run A loaded: 8160 arm-rows across 65 trials
Mapping table: 69 field definitions (8 needs_discussion_* pipeline helper fields excluded from all analyses)


---
## Tier 1: Prior Extraction Comparison

Compares the LLM extraction against the pre-existing manual extraction
(`prior_extraction_clean.csv`) on fields marked `A_reference` and
`B_triangulation` in `field_validation_mapping.csv`.

**Field selection:** 15 of 69 mapping fields are eligible for Tier 1. The
remaining 54 are either `EXCLUDED` (structural identifiers or provenance
fields), `EXCLUDED` by type (free-text only, no comparator), or marked
`no` (never collected in the prior extraction). Eligibility is determined
by the `tier1_prior_extraction` column in the mapping file.

**Group A (14 high-confidence fields):** `n`, `diagnosis`, `gender_pct_female`,
`age_mean`, `age_sd`, `age_se`, `age_other`, `time_total_days`, `fev1_pct_mean`,
`fev1_pct_sd`, `digital_literacy`, `ses`, `educational_level`, `ethnicity`

**Group B (1 triangulation field):** `healthcare_setting`

The manual extraction covers 64 of 65 trials (one excluded: cov_nr=5108,
\>30% non-COPD participants).


In [2]:
prior = load_prior_extraction(PRIOR_PATH, mapping)
print(
    f"Prior extraction loaded: {len(prior)} values",
    f"{prior['cov_nr'].nunique()} trials"
)

eligible = mapping[mapping['tier1_prior_extraction'].isin(['A_reference', 'B_triangulation'])]
print(f"Eligible fields: {len(eligible)}")
print(f"  A_reference: {len(eligible[eligible['tier1_prior_extraction'] == 'A_reference'])}")
print(f"  B_triangulation: {len(eligible[eligible['tier1_prior_extraction'] == 'B_triangulation'])}")
print("\nEligible fields for Tier 1:")
eligible_print = eligible[['tier1_prior_extraction', 'data_type_bucket']].reset_index()
print(eligible_print[['field_name', 'tier1_prior_extraction', 'data_type_bucket']].to_string())

Prior extraction loaded: 1876 values 64 trials
Eligible fields: 14
  A_reference: 14
  B_triangulation: 0

Eligible fields for Tier 1:
           field_name tier1_prior_extraction         data_type_bucket
0                   n            A_reference                numerical
1     time_total_days            A_reference                numerical
2           diagnosis            A_reference  structured_string_array
3   gender_pct_female            A_reference                numerical
4            age_mean            A_reference                numerical
5              age_sd            A_reference                numerical
6              age_se            A_reference                numerical
7           age_other            A_reference                free_text
8       fev1_pct_mean            A_reference                numerical
9         fev1_pct_sd            A_reference                numerical
10   digital_literacy            A_reference                  boolean
11                ses    

In [3]:
tier1_results = []
tier1_raw = {}
tier1_one_na_detail = []

for field_name, row in eligible.iterrows():
    bucket = row['data_type_bucket']
    if bucket == 'EXCLUDED':
        continue

    a_vals = (
            llm_run[llm_run['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']].copy()
    )
    a_vals['cov_nr'] = a_vals['cov_nr'].astype(str).str.strip().str.zfill(4)
    a_vals = a_vals.rename(columns={'value': 'value_a'})

    b_vals = (
            prior[prior['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']].copy()
    )
    b_vals['cov_nr'] = b_vals['cov_nr'].astype(str).str.strip().str.zfill(4)
    b_vals = b_vals.rename(columns={'value': 'value_b'})

    merged = a_vals.merge(b_vals, on=['cov_nr', 'arm'], how='inner')
    if merged.empty:
        continue

    paired = []
    for _, mr in merged.iterrows():
        val_a = mr['value_a']
        val_b = mr['value_b']
        # Boolean recoding for fields compared as "reported vs NA" in Tier 1
        TIER1_REPORT_ONLY = ('educational_level', 'ethnicity', 'health_literacy')
        if field_name in TIER1_REPORT_ONLY:
            # Prior: any non-empty text -> True
            if isinstance(val_b, str) and val_b.strip().lower() not in ('na', 'n/a', 'nan', ''):
                val_b = True
            elif isinstance(val_b, float) and pd.isna(val_b):
                val_b = False
            else:
                val_b = False
            # LLM: list -> any non-NA content; int -> non-zero; string -> non-empty
            if isinstance(val_a, list):
                non_na = [x for x in val_a if str(x).strip().upper() not in ('NA', 'N/A', '')]
                val_a = len(non_na) > 0
            elif isinstance(val_a, (int, float)):
                val_a = val_a != 0
            elif isinstance(val_a, str) and val_a.strip().lower() not in ('na', 'n/a', 'nan', ''):
                val_a = True
            else:
                val_a = False
            bucket = 'boolean'
        if bucket == 'structured_string_array':
            if field_name in ('educational_level', 'ethnicity') and isinstance(val_b, str):
                val_b = parse_prior_text(val_b, field_name)
            elif isinstance(val_b, str):
                val_b = [val_b]
            if isinstance(val_a, str):
                val_a = [val_a]
        if bucket == 'boolean' and field_name not in TIER1_REPORT_ONLY:
            # Prior stores verbose text for boolean fields (e.g. "Regular Internet users (%) 20")
            # Treat non-empty text as True, empty/missing as False
            if isinstance(val_b, str) and val_b.strip().lower() not in ('na', 'n/a', 'nan', ''):
                val_b = True
            else:
                val_b = False
            # LLM may store educational_level/ethnicity as lists (e.g. ['NA'] or ['High school: 40.9%'])
            # Convert to boolean: non-empty list with non-NA content = True, else False
            if isinstance(val_a, list):
                non_na = [x for x in val_a if str(x).strip().upper() not in ('NA', 'N/A', '')]
                val_a = len(non_na) > 0
        norm_a = normalize_value(val_a, bucket)
        norm_b = normalize_value(val_b, bucket)
        is_na_a = norm_a is None
        is_na_b = norm_b is None
        if is_na_a != is_na_b:
            tier1_one_na_detail.append({
                'field_name': field_name,
                'cov_nr': str(mr['cov_nr']).strip(),
                'arm': str(mr['arm']).strip(),
                'llm_extraction_is_na': is_na_a,
                'human_extraction_is_na': is_na_b,
                'llm_extraction_value_raw': str(mr['value_a'])[:80],
                'human_extraction_value_raw': str(mr['value_b'])[:80],
            })
        paired.append((norm_a, norm_b, str(mr['cov_nr']).strip(), str(mr['arm']).strip()))

    result = compute_agreement(field_name, bucket, paired)
    result.extra = {'tier1_group': 'A' if row['tier1_prior_extraction'] == 'A_reference' else 'B'}
    tier1_results.append(result)
    if result.raw_disagreements:
        tier1_raw[field_name] = result.raw_disagreements

print(f"Tier 1 analyzed: {len(tier1_results)} fields")

Tier 1 analyzed: 14 fields


In [4]:
rows = []
for r in tier1_results:
    n_total = r.n_compared + r.n_both_na + r.n_one_na
    rows.append({
        'field_name': r.field_name,
        'bucket': r.bucket,
        'tier1_group': r.extra.get('tier1_group') if r.extra else '',
        'n_total': n_total,
        'n_compared': r.n_compared,
        'n_both_na': r.n_both_na,
        'n_one_na': r.n_one_na,
        'primary_metric': r.primary_metric_name,
        'primary_value': round(r.primary_metric_value, 4) if r.primary_metric_value is not None else None,
        'secondary_metric': r.secondary_metric_name,
        'secondary_value': round(r.secondary_metric_value, 4) if r.secondary_metric_value is not None else None,
        'tertiary_metric': r.tertiary_metric_name,
        'tertiary_value': round(r.tertiary_metric_value, 4) if r.tertiary_metric_value is not None else None,
        'flagged': r.flagged,
        'flag_reason': r.flag_reason if r.flagged else '',
    })
tier1_per_field = pd.DataFrame(rows)
# Add one-NA cov_nr columns from the detail collector
if tier1_one_na_detail:
    tier1_one_na_df = pd.DataFrame(tier1_one_na_detail)
    llm_list = tier1_one_na_df[tier1_one_na_df['llm_extraction_is_na']].groupby('field_name')['cov_nr'].apply(
        lambda x: '; '.join(sorted(x.unique())))
    prior_list = tier1_one_na_df[tier1_one_na_df['human_extraction_is_na']].groupby('field_name')['cov_nr'].apply(
        lambda x: '; '.join(sorted(x.unique())))
    tier1_per_field = tier1_per_field.merge(
        llm_list.rename('one_na_llm_cov_nrs').to_frame(), on='field_name', how='left')
    tier1_per_field = tier1_per_field.merge(
        prior_list.rename('one_na_prior_cov_nrs').to_frame(), on='field_name', how='left')
    tier1_per_field['one_na_llm_cov_nrs'] = tier1_per_field['one_na_llm_cov_nrs'].fillna('')
    tier1_per_field['one_na_prior_cov_nrs'] = tier1_per_field['one_na_prior_cov_nrs'].fillna('')
else:
    tier1_per_field['one_na_llm_cov_nrs'] = ''
    tier1_per_field['one_na_prior_cov_nrs'] = ''
print(
    f"Fields: {len(tier1_per_field)}",
    f"Flagged: {tier1_per_field['flagged'].sum()}"
)

tier1_per_field.to_csv(OUTPUT_DIR / 'pipeline' / 'tier1_per_field.csv',
        index=False)
print("Saved: tier1_per_field.csv")

Fields: 14 Flagged: 8


Saved: tier1_per_field.csv


In [5]:
# Generate adjudication queue for Group A disagreements
group_a_field_names = set(eligible[eligible['tier1_prior_extraction'] == 'A_reference'].index)
queue_rows = []
for field_name, disagreements in tier1_raw.items():
    if field_name not in group_a_field_names:
        continue
    for d in disagreements:
        queue_rows.append({
            'field_name': field_name,
            'cov_nr': d.get('cov_nr'),
            'arm': d.get('arm'),
            'llm_value': d.get('value_a'),
            'prior_value': d.get('value_b'),
            'adjudication': '',
            'adjudicator_note': '',
        })

queue_df = pd.DataFrame(queue_rows)
queue_df.to_csv(OUTPUT_DIR / 'pipeline' / 'tier1_adjudication_queue.csv',
        index=False)
# n_unique counts only fields with actual disagreements — may be <14 A_reference fields
print(f"Adjudication queue: {len(queue_df)} disagreements across {queue_df['field_name'].nunique()} A_reference fields")
print("Saved: tier1_adjudication_queue.csv")

Adjudication queue: 219 disagreements across 12 A_reference fields
Saved: tier1_adjudication_queue.csv


In [6]:
# Bucket summary
bucket_groups = tier1_per_field.groupby('bucket')
for bucket, group in bucket_groups:
    flagged = group['flagged'].sum()
    primary_vals = group['primary_value'].dropna()
    if len(primary_vals) > 0:
        print(f"  {bucket}: {len(group)} fields, {flagged} flagged, "
              f"median {group.iloc[0]['primary_metric']}={primary_vals.median():.3f}")
    else:
        print(f"  {bucket}: {len(group)} fields, {flagged} flagged (no primary values)")

  boolean: 4 fields, 1 flagged, median gwet_ac1=0.942
  free_text: 1 fields, 0 flagged, median token_f1=0.141
  numerical: 8 fields, 7 flagged, median icc_2_1=0.970
  structured_string_array: 1 fields, 0 flagged, median key_jaccard=0.974


In [7]:
# One-NA detail — cases where exactly one side (LLM or prior) normalized to None
if tier1_one_na_detail:
    one_na_df = pd.DataFrame(tier1_one_na_detail)
    print(f"Total one-NA cases: {len(one_na_df)}")
    print()
    for field_name, field_one_na_cases in one_na_df.groupby('field_name'):
        na_llm = field_one_na_cases['llm_extraction_is_na'].sum()
        na_prior = field_one_na_cases['human_extraction_is_na'].sum()
        print(f"  {field_name}: {len(field_one_na_cases)} cases (LLM extraction NA={na_llm}, Human extraction NA={na_prior})")
        llm_na_nrs = field_one_na_cases[field_one_na_cases['llm_extraction_is_na']]['cov_nr'].unique()[:3]
        prior_na_nrs = (
                field_one_na_cases[field_one_na_cases['human_extraction_is_na']]
                ['cov_nr'].unique()[:3]
        )
        if len(llm_na_nrs):
            print(f"    LLM extraction NA cov_nrs: {', '.join(llm_na_nrs)}")
        if len(prior_na_nrs):
            print(f"    Human extraction NA cov_nrs: {', '.join(prior_na_nrs)}")
    print()
    print("Sample raw values (first 5 rows):")
    print(one_na_df.head(10).to_string(index=False))
else:
    print("No one-NA cases found.")


Total one-NA cases: 26

  age_mean: 4 cases (LLM extraction NA=4, Human extraction NA=0)
    LLM extraction NA cov_nrs: 1288, 1800
  age_other: 6 cases (LLM extraction NA=0, Human extraction NA=6)
    Human extraction NA cov_nrs: 0464, 1203, 1800
  age_sd: 2 cases (LLM extraction NA=2, Human extraction NA=0)
    LLM extraction NA cov_nrs: 1800
  fev1_pct_mean: 4 cases (LLM extraction NA=4, Human extraction NA=0)
    LLM extraction NA cov_nrs: 1800, 4337
  fev1_pct_sd: 2 cases (LLM extraction NA=2, Human extraction NA=0)
    LLM extraction NA cov_nrs: 1800
  gender_pct_female: 4 cases (LLM extraction NA=4, Human extraction NA=0)
    LLM extraction NA cov_nrs: 0157, 1800
  time_total_days: 4 cases (LLM extraction NA=4, Human extraction NA=0)
    LLM extraction NA cov_nrs: 1400, 4986

Sample raw values (first 5 rows):
       field_name cov_nr     arm  llm_extraction_is_na  human_extraction_is_na llm_extraction_value_raw human_extraction_value_raw
  time_total_days   1400 control          

---
## Tier 2: 5-Trial Gold Standard Comparison

Full-schema comparison (61 non-excluded fields) between the LLM
extraction and a manually re-extracted gold standard of 5 trials
(1228, 1800, 2003, 2667, 4337).

**One-NA tracking:** When one side has a value and the other returns NA,
the case is collected in `tier2_one_na_detail` for per-field audit. This
is separate from N01's inter-run one-NA analysis — here the discrepancy
is LLM vs human, not LLM run A vs run B.

**Undersampled flag:** Fields with fewer than 5 comparable pairs
(`n_compared < 5`) are marked as undersampled — the 5-trial sample size
limits what can be concluded.

**Adjudication queue:** Disagreements between the LLM
extraction and the 5-trial gold standard are compiled into an
adjudication queue (`tier2_adjudication_queue.csv`) for human
review. Unlike Tier 1 (which filters to Group A fields only),
Tier 2 includes disagreements for all non-excluded fields —
the gold standard sample is small enough (5 trials) to review
comprehensively.


In [8]:
# Load 5-trial gold standard (Tier 2: full-schema manual re-extraction).
# Each trial was independently re-extracted by a human annotator covering
# the full 69-field schema. These 5 trials form the reference truth for
# comparing LLM extraction accuracy.
gold = load_gold_standard(GOLD_STANDARD_PATH)
print(
    f"Gold standard loaded: {len(gold)} values",
    f"{gold['cov_nr'].nunique()} trials"
)
print(f"Gold standard trials: {sorted(gold['cov_nr'].unique())}")

Gold standard loaded: 590 values 5 trials
Gold standard trials: ['1228', '1800', '2003', '2667', '4337']


In [9]:
# Tier 2: Full-schema comparison on 5 manually re-extracted trials.
# For each non-excluded field, pair LLM values against the gold standard
# on matching (cov_nr, arm). Normalise raw values using bucket-specific
# rules, then compute agreement metrics via compute_agreement().
#
# Structured-string-array fields (diagnosis, smoking_status, etc.) need
# special handling: the gold standard CSV stored them as plain strings
# (CSV quoting artifact), so we parse them into structured lists before
# normalisation.
#
# n_compared counts pairs where both sides have a value. n_one_na counts
# pairs where exactly one side is NA. n_both_na counts pairs where both
# are NA (not comparable, excluded from agreement computation).
# Undersampled flag = n_compared < 5.
tier2_results = []
tier2_raw = {}
tier2_one_na_detail = []

for field_name, row in mapping.iterrows():
    bucket = row['data_type_bucket']
    if bucket == 'EXCLUDED':
        continue

    a_vals = (
            llm_run[llm_run['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']].copy()
    )
    a_vals['cov_nr'] = a_vals['cov_nr'].astype(str).str.strip()
    a_vals = a_vals.rename(columns={'value': 'value_a'})

    b_vals = (
            gold[gold['field_name'] == field_name]
            [['cov_nr', 'arm', 'value']].copy()
    )
    b_vals['cov_nr'] = b_vals['cov_nr'].astype(str).str.strip()
    b_vals = b_vals.rename(columns={'value': 'value_b'})

    merged = a_vals.merge(b_vals, on=['cov_nr', 'arm'], how='inner')
    if merged.empty:
        continue

    paired = []
    for _, mr in merged.iterrows():
        val_a = mr['value_a']
        val_b = mr['value_b']
        if bucket == 'structured_string_array':
            if isinstance(val_b, str):
                parsed = parse_structured_array(val_b)
                val_b = parsed if parsed is not None else [val_b]
            if isinstance(val_a, str):
                parsed = parse_structured_array(val_a)
                val_a = parsed if parsed is not None else [val_a]
        norm_a = normalize_value(val_a, bucket)
        norm_b = normalize_value(val_b, bucket)
        is_na_a = norm_a is None
        is_na_b = norm_b is None
        if is_na_a != is_na_b:
            tier2_one_na_detail.append({
                'field_name': field_name,
                'cov_nr': str(mr['cov_nr']).strip(),
                'arm': str(mr['arm']).strip(),
                'llm_extraction_is_na': is_na_a,
                'human_extraction_is_na': is_na_b,
                'llm_extraction_value_raw': str(mr['value_a'])[:80],
                'human_extraction_value_raw': str(mr['value_b'])[:80],
            })
        paired.append((norm_a, norm_b, str(mr['cov_nr']).strip(), str(mr['arm']).strip()))

    result = compute_agreement(field_name, bucket, paired)
    result.extra = {'undersampled': result.n_compared < 5}
    tier2_results.append(result)
    if result.raw_disagreements:
        tier2_raw[field_name] = result.raw_disagreements

gold_arms = gold[['cov_nr', 'arm']].drop_duplicates()
print(f"Tier 2 analyzed: {len(tier2_results)} fields across {gold['cov_nr'].nunique()} trials ({len(gold_arms)} arms)")


Tier 2 analyzed: 59 fields across 5 trials (10 arms)


In [10]:
# Build Tier 2 per-field results table. Columns mirror Tier 1 structure with
# an added 'undersampled' flag. One-NA detail columns appended from the
# tier2_one_na_detail collector — same pattern as Tier 1.
rows = []
for r in tier2_results:
    n_total = r.n_compared + r.n_both_na + r.n_one_na
    undersampled = r.extra.get('undersampled', False) if r.extra else False
    rows.append({
        'field_name': r.field_name,
        'bucket': r.bucket,
        'n_total': n_total,
        'n_compared': r.n_compared,
        'n_both_na': r.n_both_na,
        'n_one_na': r.n_one_na,
        'undersampled': undersampled,
        'primary_metric': r.primary_metric_name,
        'primary_value': round(r.primary_metric_value, 4) if r.primary_metric_value is not None else None,
        'secondary_metric': r.secondary_metric_name,
        'secondary_value': round(r.secondary_metric_value, 4) if r.secondary_metric_value is not None else None,
        'tertiary_metric': r.tertiary_metric_name,
        'tertiary_value': round(r.tertiary_metric_value, 4) if r.tertiary_metric_value is not None else None,
        'flagged': r.flagged,
        'flag_reason': r.flag_reason if r.flagged else '',
    })
tier2_per_field = pd.DataFrame(rows)
# Add one-NA cov_nr columns from the detail collector
if tier2_one_na_detail:
    tier2_one_na_df = pd.DataFrame(tier2_one_na_detail)
    llm_list = tier2_one_na_df[tier2_one_na_df['llm_extraction_is_na']].groupby('field_name')['cov_nr'].apply(
        lambda x: '; '.join(sorted(x.unique())))
    gold_list = tier2_one_na_df[tier2_one_na_df['human_extraction_is_na']].groupby('field_name')['cov_nr'].apply(
        lambda x: '; '.join(sorted(x.unique())))
    tier2_per_field = tier2_per_field.merge(
        llm_list.rename('one_na_llm_cov_nrs').to_frame(), on='field_name', how='left')
    tier2_per_field = tier2_per_field.merge(
        gold_list.rename('one_na_gold_cov_nrs').to_frame(), on='field_name', how='left')
    tier2_per_field['one_na_llm_cov_nrs'] = tier2_per_field['one_na_llm_cov_nrs'].fillna('')
    tier2_per_field['one_na_gold_cov_nrs'] = tier2_per_field['one_na_gold_cov_nrs'].fillna('')
else:
    tier2_per_field['one_na_llm_cov_nrs'] = ''
    tier2_per_field['one_na_gold_cov_nrs'] = ''
print(
    f"Fields: {len(tier2_per_field)}, Flagged: {tier2_per_field['flagged'].sum()}, "
      f"Undersampled: {tier2_per_field['undersampled'].sum()}")

tier2_per_field.to_csv(OUTPUT_DIR / 'pipeline' / 'tier2_per_field.csv',
        index=False)
print("Saved: tier2_per_field.csv")

Fields: 59, Flagged: 13, Undersampled: 34
Saved: tier2_per_field.csv


In [11]:
# Build Tier 2 adjudication queue. Unlike Tier 1 (which filters to Group A
# only), Tier 2 includes disagreements for all fields — the gold standard
# is small enough (5 trials) to review comprehensively.
queue_rows = []
for field_name, disagreements in tier2_raw.items():
    for d in disagreements:
        queue_rows.append({
            'field_name': field_name,
            'cov_nr': d.get('cov_nr'),
            'arm': d.get('arm'),
            'llm_value': d.get('value_a'),
            'gold_value': d.get('value_b'),
            'adjudication': '',
            'adjudicator_note': '',
        })

if queue_rows:
    queue_df = pd.DataFrame(queue_rows)
    queue_df.to_csv(OUTPUT_DIR / 'pipeline' / 'tier2_adjudication_queue.csv',
            index=False)
    print(f"Tier 2 adjudication queue: {len(queue_df)} disagreements across "
          f"{queue_df['field_name'].nunique()} fields")
    print("Saved: tier2_adjudication_queue.csv")
else:
    print("Tier 2: no disagreements found — queue is empty")
    pd.DataFrame(columns=['field_name', 'cov_nr', 'arm', 'llm_value', 'gold_value',
                          'adjudication', 'adjudicator_note']
                ).to_csv(OUTPUT_DIR / 'pipeline' / 'tier2_adjudication_queue.csv', index=False)
    print("Saved: tier2_adjudication_queue.csv (empty)")

Tier 2 adjudication queue: 76 disagreements across 18 fields
Saved: tier2_adjudication_queue.csv


In [12]:
# Print Tier 2 flagged fields — low primary metric value or undersampled.
# Sorted by primary_value ascending (worst first).
flagged2 = tier2_per_field[tier2_per_field['flagged']].sort_values('primary_value')
if not flagged2.empty:
    print("Flagged fields:")
    for _, flagged_row in flagged2.iterrows():
        undersampled_marker = " [UNDERSAMPLED]" if flagged_row['undersampled'] else ""
        print(f"  {flagged_row['field_name']:<40} {flagged_row['primary_metric']}={flagged_row['primary_value']:.3f}{undersampled_marker}")
else:
    print("No fields flagged")

Flagged fields:
  time_intervention_days                   icc_2_1=-0.309
  diagnosis                                key_jaccard=0.000
  smoking_status                           key_jaccard=0.000 [UNDERSAMPLED]
  disease_severity_other                   key_jaccard=0.000
  healthcare_setting_confidence            gwet_ac1=0.010
  time_followup_days                       icc_2_1=0.187
  time_total_days                          icc_2_1=0.577
  digital_strategy_provides_ongoing_support gwet_ac1=0.597
  digital_strategy_provides_equipment      gwet_ac1=0.655
  healthcare_setting                       gwet_ac1=0.756
  ses                                      gwet_ac1=0.756
  bmi_sd                                   icc_2_1=1.000 [UNDERSAMPLED]
  bmi_mean                                 icc_2_1=1.000 [UNDERSAMPLED]


---
## Tier 3: 10-Trial Spot Check

Summarizes the qualitative error review from `tier3_filled.csv`.
The spot check covers 10 trials with error types:
- `no_error`: correct extraction
- `minor_error`: slight mistake (rule misapplication)
- `major_error`: wrong extraction (hallucinated value)

Correction notes are extracted for A_reference and B_triangulation fields.

**Relevant fields:** Of the 69 field definitions in the mapping table, 49 are actual extraction fields. The remaining 20 are excluded: structural identifiers (`cov_nr`, `arm`), `_error_type` columns themselves, and `_correction_note` columns. Only the 49 extraction fields contribute to the error summary below.

In [13]:
tier3 = pd.read_csv(TIER3_PATH)
print(f"Tier 3 loaded: {len(tier3)} arms, {tier3['cov_nr'].nunique() if 'cov_nr' in tier3.columns else 'N/A'} trials")

# Summarize error frequencies across all field-specific error_type columns
error_cols = [c for c in tier3.columns if c.endswith('_error_type') and c != '_metadata_error_type']
print(f"Fields with error classifications: {len(error_cols)}")

# Relevant fields (49 extraction fields, excluding auxiliary columns)
RELEVANT_FIELDS = {
    'age_mean', 'age_other', 'age_sd', 'age_se',
    'bmi_mean', 'bmi_other', 'bmi_sd',
    'bp_diastolic_mean', 'bp_diastolic_sd', 'bp_other',
    'bp_systolic_mean', 'bp_systolic_sd',
    'diagnosis',
    'digital_literacy', 'digital_literacy_frequency',
    'digital_literacy_possession', 'digital_literacy_skills',
    'digital_strategy_excludes',
    'digital_strategy_provides_equipment',
    'digital_strategy_provides_ongoing_support',
    'digital_strategy_provides_training',
    'educational_level', 'ethnicity',
    'fev1_other', 'fev1_pct_mean', 'fev1_pct_sd',
    'gender_female_n', 'gender_male_n',
    'gender_pct_female', 'gender_pct_male',
    'health_literacy', 'health_literacy_instrument_name',
    'health_literacy_instrument_other', 'health_literacy_instrument_value',
    'healthcare_setting',
    'n',
    'pack_years_mean', 'pack_years_other', 'pack_years_sd',
    'ses', 'ses_income', 'ses_job_status',
    'ses_living_location', 'ses_living_situation',
    'ses_relationship_status',
    'smoking_status',
    'time_total_days',
}

Tier 3 loaded: 21 arms, 10 trials
Fields with error classifications: 69


In [14]:
# Rename error types: none→no_error, rule_misapplication→minor_error, hallucinated_value→major_error
ERROR_TYPE_MAP = {
    'none': 'no_error',
    'rule_misapplication': 'minor_error',
    'hallucinated_value': 'major_error',
}

# Filter to relevant fields only
relevant_error_cols = [c for c in error_cols
                       if c.replace('_error_type', '') in RELEVANT_FIELDS]
print(f"Relevant fields with error classifications: {len(relevant_error_cols)}")

error_counts = {}
for col in relevant_error_cols:
    counts = tier3[col].value_counts()
    for error_type, count in counts.items():
        mapped = ERROR_TYPE_MAP.get(error_type, error_type)
        if mapped not in error_counts:
            error_counts[mapped] = 0
        error_counts[mapped] += count

error_summary = pd.DataFrame([
    {'error_type': k, 'count': v, 'pct': v / sum(error_counts.values()) * 100}
    for k, v in sorted(error_counts.items(), key=lambda x: -x[1])
])

# Arm-level counts per error type
n_arms = len(tier3)
arm_has_error = tier3[relevant_error_cols].apply(
    lambda row: any(ERROR_TYPE_MAP.get(v, v) != 'no_error' for v in row), axis=1)
n_arms_any_error = arm_has_error.sum()

for mapped_type, orig_type in [('no_error', 'none'),
                                ('minor_error', 'rule_misapplication'),
                                ('major_error', 'hallucinated_value')]:
    if mapped_type == 'no_error':
        arm_mask = ~(tier3[relevant_error_cols].isin(['rule_misapplication', 'hallucinated_value'])).any(axis=1)
    else:
        arm_mask = (tier3[relevant_error_cols] == orig_type).any(axis=1)
    n_arms_type = arm_mask.sum()
    error_summary.loc[error_summary['error_type'] == mapped_type,
                      'n_arms_with_type'] = n_arms_type

error_summary['n_arms_with_type'] = error_summary['n_arms_with_type'].fillna(0).astype(int)
error_summary['n_arms_total'] = n_arms

# Append a row for arms_with_any_error
any_row = pd.DataFrame([{
    'error_type': 'arms_with_any_error',
    'count': n_arms_any_error,
    'pct': n_arms_any_error / n_arms * 100,
    'n_arms_with_type': n_arms_any_error,
    'n_arms_total': n_arms,
}])
error_summary = pd.concat([error_summary, any_row], ignore_index=True)

print(f"\nArms with any error: {n_arms_any_error} / {n_arms}")

print("\nError type distribution (relevant fields only):")
print(error_summary.to_string(index=False))
error_summary.to_csv(OUTPUT_DIR / 'pipeline' / 'tier3_error_summary.csv',
        index=False)
print("\nSaved: tier3_error_summary.csv")


Relevant fields with error classifications: 47

Arms with any error: 18 / 21

Error type distribution (relevant fields only):
         error_type  count       pct  n_arms_with_type  n_arms_total
           no_error    931 94.326241                 3            21
        minor_error     49  4.964539                17            21
        major_error      7  0.709220                 7            21
arms_with_any_error     18 85.714286                18            21

Saved: tier3_error_summary.csv


In [15]:
# Per-field error breakdown (relevant fields only)
relevant_error_cols = [c for c in error_cols
                       if c.replace('_error_type', '') in RELEVANT_FIELDS]

per_field_rows = []
for col in relevant_error_cols:
    field_name = col.replace('_error_type', '')
    val_col = field_name
    n_arms = len(tier3)
    n_reported = tier3[val_col].notna().sum() if val_col in tier3.columns else 0
    n_na = n_arms - n_reported
    # Error type counts with renamed labels
    err_counts = tier3[col].value_counts()
    row = {
        'field_name': field_name,
        'n_arms': n_arms,
        'n_reported': n_reported,
        'n_na': n_na,
        'n_no_error': err_counts.get('none', 0),
        'n_minor_error': err_counts.get('rule_misapplication', 0),
        'n_major_error': err_counts.get('hallucinated_value', 0),
    }
    per_field_rows.append(row)

tier3_by_field = pd.DataFrame(per_field_rows)
tier3_by_field = tier3_by_field.sort_values('field_name')

# Add total_errors column
tier3_by_field['n_any_error'] = (tier3_by_field['n_minor_error']
                                 + tier3_by_field['n_major_error'])

tier3_by_field.to_csv(OUTPUT_DIR / 'pipeline' / 'tier3_error_by_field.csv',
        index=False)
print("Saved: tier3_error_by_field.csv")

# Show fields with errors
has_errors = tier3_by_field[tier3_by_field['n_any_error'] > 0]
print(f"\nFields with errors ({len(has_errors)} fields):")
print(
    f"(reported = arms where the LLM extracted a non-NA value",
    f"out of {len(tier3)} arms)"
)
for _, row in has_errors.iterrows():
    print(f"  {row['field_name']:<45} "
          f"minor={row['n_minor_error']} major={row['n_major_error']} "
          f"reported={row['n_reported']}/{row['n_arms']}")


Saved: tier3_error_by_field.csv

Fields with errors (13 fields):
(reported = arms where the LLM extracted a non-NA value out of 21 arms)
  digital_literacy_skills                       minor=3 major=0 reported=3/21
  digital_strategy_provides_equipment           minor=0 major=4 reported=21/21
  digital_strategy_provides_training            minor=1 major=0 reported=21/21
  educational_level                             minor=4 major=2 reported=6/21
  ethnicity                                     minor=4 major=0 reported=8/21
  healthcare_setting                            minor=0 major=1 reported=21/21
  n                                             minor=4 major=0 reported=21/21
  ses                                           minor=7 major=0 reported=21/21
  ses_income                                    minor=2 major=0 reported=2/21
  ses_living_location                           minor=2 major=0 reported=2/21
  ses_living_situation                          minor=9 major=0 reported=6/21


In [16]:
# List correction notes for A_reference and B_triangulation fields
note_cols = [c for c in tier3.columns if c.endswith('_correction_note')]
if note_cols:
    print("Correction notes:")
    for col in note_cols:
        vals = tier3[col].dropna()
        if len(vals) > 0:
            field = col.replace('_correction_note', '')
            print(f"\n  {field} ({len(vals)} corrections):")
            for _, row in tier3[tier3[col].notna()].iterrows():
                cov = row.get('cov_nr', '?')
                arm = row.get('arm', '?')
                note = row[col]
                error_col_name = field + '_error_type'
                raw_err = row.get(error_col_name, '?') if error_col_name in tier3.columns else '?'
                err_label = ERROR_TYPE_MAP.get(raw_err, raw_err)
                note_str = str(note) if not isinstance(note, str) else note
                print(f"    [{cov}/{arm}] [{err_label}] {note_str[:100]}")


Correction notes:

  educational_level (6 corrections):
    [469/treat1] [minor_error] percentage extracted correctly, n reported but not extracted. Weird answers when recalculating n fro
    [469/control] [minor_error] percentage extracted correctly, n reported but not extracted. Weird answers when recalculating n fro
    [1952/treat1] [minor_error] recalculated percentages, more accurate than reported! No n reported, but possible to calculate.
    [1952/control] [minor_error] recalculated percentages, more accurate than reported! No n reported, but possible to calculate.
    [4727/treat1] [major_error] reported as "Level of education" - avg decimal where: 1=GCSE/Olevel, 2= Alevel/HNC, 3=uni, 4=grad
    [4727/control] [major_error] reported as "Level of education" - avg decimal where: 1=GCSE/Olevel, 2= Alevel/HNC, 3=uni, 4=grad

  ethnicity (6 corrections):
    [469/treat1] [minor_error] percentages extracted correctly, n reported but not extracted. Weird answers when recalculating n 

---
## Combined Summary for Appendix 2

Consolidated results across all three tiers.

In [17]:
summary_rows = []

# Tier 1 summary
tier1_group_a_fields = tier1_per_field[tier1_per_field['tier1_group'] == 'A']
tier1_group_b_fields = tier1_per_field[tier1_per_field['tier1_group'] == 'B']
summary_rows.append({
    'tier': 'Tier 1 (Group A)',
    'n_fields': len(tier1_group_a_fields),
    'n_flagged': int(tier1_group_a_fields['flagged'].sum()),
    'n_trials': prior['cov_nr'].nunique(),
    'note': 'Corpus-wide prior extraction, high-confidence fields',
})
summary_rows.append({
    'tier': 'Tier 1 (Group B)',
    'n_fields': len(tier1_group_b_fields),
    'n_flagged': int(tier1_group_b_fields['flagged'].sum()),
    'n_trials': prior['cov_nr'].nunique(),
    'note': 'Corpus-wide prior extraction, triangulation only',
})

# Tier 2 summary
n_t2_flagged = int(tier2_per_field['flagged'].sum())
n_t2_undersampled = int(tier2_per_field['undersampled'].sum())
summary_rows.append({
    'tier': 'Tier 2',
    'n_fields': len(tier2_per_field),
    'n_flagged': n_t2_flagged,
    'n_trials': gold['cov_nr'].nunique(),
    'note': f'5-trial gold standard; {n_t2_undersampled} fields undersampled (n<5)',
})

# Tier 3 summary
non_none_count = sum(v for k, v in error_counts.items() if k != 'no_error')
total_cells = sum(error_counts.values())
summary_rows.append({
    'tier': 'Tier 3',
    'n_fields': len(relevant_error_cols),
    'n_flagged': non_none_count,
    'n_trials': tier3['cov_nr'].nunique() if 'cov_nr' in tier3.columns else 10,
    'note': f'{non_none_count}/{total_cells} cells with errors ({non_none_count/total_cells*100:.1f}%)',
})

combined = pd.DataFrame(summary_rows)
combined.to_csv(OUTPUT_DIR / 'pipeline' / 'appendix2_combined.csv',
        index=False)
print("Combined summary:")
print(combined.to_string(index=False))
print("\nSaved: appendix2_combined.csv")

Combined summary:
            tier  n_fields  n_flagged  n_trials                                                 note
Tier 1 (Group A)        14          8        64 Corpus-wide prior extraction, high-confidence fields
Tier 1 (Group B)         0          0        64     Corpus-wide prior extraction, triangulation only
          Tier 2        59         13         5  5-trial gold standard; 34 fields undersampled (n<5)
          Tier 3        47         56        10                      56/987 cells with errors (5.7%)

Saved: appendix2_combined.csv


In [18]:
print("=== Notebook 02 complete ===")
print(f"Completed at: {pd.Timestamp.now().isoformat()}")
print(
    f"Tier 1: {len(tier1_per_field)} fields",
    f"{int(tier1_per_field['flagged'].sum())} flagged"
)
print(
    f"Tier 2: {len(tier2_per_field)} fields",
    f"{int(tier2_per_field['flagged'].sum())} flagged"
)
print(f"Tier 3: {len(relevant_error_cols)} fields, {non_none_count} error cells")

print(f"Schema hash: 2b702021 (this revision's reference)")
# Schema staleness check: compare computed hash against stored version.
# If column schemas changed since last run, downstream notebooks may
# produce stale results.
from src.analysis.data_loading import get_schema_hash
current_hash = get_schema_hash()
version_file = ROOT / "data" / "processed" / "schema_version.txt"
if version_file.exists():
    stored_hash = version_file.read_text().strip()
    if current_hash != stored_hash:
        print(f"WARNING: Schema hash changed from {stored_hash} to {current_hash} — downstream data may be stale")
    else:
        print(f"Schema hash verified: {current_hash}")
else:
    print(f"Schema hash: {current_hash} (first run — no prior hash stored)")
version_file.write_text(current_hash)


=== Notebook 02 complete ===
Completed at: 2026-06-14T17:06:12.601972
Tier 1: 14 fields 8 flagged
Tier 2: 59 fields 13 flagged
Tier 3: 47 fields, 56 error cells
Schema hash: 2b702021 (this revision's reference)
Schema hash verified: 2b702021


8